# Project UAS Machine Learning

**Kelompok:** [Isi Nama Kelompok/Anggota di Sini]

Sesuai instruksi UAS, di notebook ini kita membuat *pipeline machine learning* untuk ngulik dua jenis data dengan karakteristik yang beda-beda banget:
1. **Data MADELON**: Ini tipikal *high-dimensional data* (banyak banget fiturnya).
2. **Data Credit Card Fraud**: Ini tipikal data yang *sangat imbalanced* (hampir semuanya masuk ke satu kategori).

Kita bakal ngerjain semuanya lengkap: dari Preprocessing, Tuning Model, sampai ngitung tingkat Error/Akurasi-nya pakai matrik evaluasi!

In [ ]:
# Pertama, kita import dulu semua library yang umum dipakai
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Library untuk bagi data dan nyari settingan algoritma terbaik
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold

# Tools untuk ngerapihin data (Preprocessing)
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.decomposition import PCA

# Kita pakai Pipeline dari imblearn karena kita mau nyisipin SMOTE
# Biar pas Cross Validation (CV) datanya gak bocor/gak berantakan
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

# List algoritma ML yang bakal kita coba pakaikan
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Fungsi-fungsi untuk menghitung nilai evaluasi
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve

# Mengabaikan error peringatan kecil dari library
import warnings
warnings.filterwarnings('ignore')

# Set visual grafik biar nggak polos-polos amat
sns.set_theme(style="whitegrid")
# Default random state biar hasilnya konsisten tiap di-run
RANDOM_SEED = 42

---
## 1. Exploratory Data Analysis (EDA)

### 1.1 Persiapan Dataset
*Catatan: Pastikan taruh file data mentahnya se-folder sama notebook ini, ya!*

In [ ]:
# ----------------------------------------------------
# A. Fungsi untuk narik data MADELON
# ----------------------------------------------------
def siapin_data_madelon():
    try:
        # File ini agak unik karena pemisahnya spasi, makanya pakai sep='\s+'
        fitur_train = pd.read_csv('madelon_train.data', header=None, sep=r'\s+')
        label_train = pd.read_csv('madelon_train.labels', header=None, sep=r'\s+')
        
        fitur_valid = pd.read_csv('madelon_valid.data', header=None, sep=r'\s+')
        label_valid = pd.read_csv('madelon_valid.labels', header=None, sep=r'\s+')
        
        # Namain kolom labelnya biar jelas
        label_train.columns = ['kelas_target']
        label_valid.columns = ['kelas_target']
        
        # Gabungin dulu antara fitur (X) dan labelnya (Y)
        df_train = pd.concat([fitur_train, label_train], axis=1)
        df_valid = pd.concat([fitur_valid, label_valid], axis=1)
        
        # Kita satuin full biar train_test_split-nya bisa kita atur sendiri dari nol
        data_lengkap = pd.concat([df_train, df_valid], axis=0).reset_index(drop=True)
        return data_lengkap
    
    except Exception as error:
        print("Yah, data MADELON gak ketarik. Pastiin nama file dan letaknya udah bener ya!", error)
        print("[INFO] Bikin data tiruan/dummy biar kode tetep bisa jalan...")
        from sklearn.datasets import make_classification
        X, y = make_classification(n_samples=2600, n_features=500, n_informative=20, random_state=RANDOM_SEED)
        df_dummy = pd.DataFrame(X)
        df_dummy['kelas_target'] = y
        df_dummy['kelas_target'] = df_dummy['kelas_target'].replace({0: -1}) 
        return df_dummy

data_madelon = siapin_data_madelon()

# ----------------------------------------------------
# B. Fungsi untuk narik data Credit Card Fraud
# ----------------------------------------------------
def siapin_data_creditcard():
    try:
        data_cc = pd.read_csv('creditcard.csv')
        return data_cc
    except Exception as error:
        print("Data Credit Card Fraud gak ketarik:", error)
        print("[INFO] Bikin data tiruan/dummy biar kodenya gak berenti...")
        from sklearn.datasets import make_classification
        X, y = make_classification(n_samples=5000, n_features=30, n_informative=10, weights=[0.99, 0.01], random_state=RANDOM_SEED)
        df_dummy_cc = pd.DataFrame(X, columns=['V'+str(i) for i in range(1, 29)] + ['Amount', 'Time'])
        df_dummy_cc['Class'] = y
        return df_dummy_cc

data_fraud = siapin_data_creditcard()

### 1.2 Mengulik Karakteristik MADELON

In [ ]:
print("=== Hasil Ceking MADELON ===")
print("Ukuran dataset kita (baris, kolom):", data_madelon.shape)
print("Total yang datanya kosong/bolong (missing values):", data_madelon.isnull().sum().sum())
print("Penyebaran datanya:\n", data_madelon['kelas_target'].value_counts())

plt.figure(figsize=(6,4))
sns.countplot(x='kelas_target', data=data_madelon, palette='Set2')
plt.title("Ini Bukti Kalau Kelas Target MADELON Itu Lumayan Seimbang")
plt.show()

# Biar gak kepanjangan, kita tampilin contoh 10 fitur pertamanya aja
display(data_madelon.iloc[:, :10].describe())

**Penjelasan Khusus Buat Data MADELON:**  
Bisa dilihat dataset ini pantes banget dijuluki *high-dimensional data*. Lha iya, satu baris observasi aja butuh nyimpen 500 kolom atribut. Kasus begini sering bikin pusing komputer (*curse of dimensionality*) karena algoritmanya bisa terjebak belajar dari derau/noise yang sebenernya nggak penting.  
Tapi untungnya, persebaran target (+1 dan -1)-nya udah lumayan adil dan seimbang.

### 1.3 Mengulik Karakteristik Credit Card Fraud

In [ ]:
print("=== Hasil Ceking Credit Card Fraud ===")
print("Ukuran dataset (baris, kolom):", data_fraud.shape)
print("Total yang datanya bolong (missing values):", data_fraud.isnull().sum().sum())
print("Penyebaran datanya (Lihat perbedaannya):\n", data_fraud['Class'].value_counts())

plt.figure(figsize=(6,4))
sns.countplot(x='Class', data=data_fraud, palette='Set1')
plt.title("Distribusi Credit Card Fraud (Pake Skala Log karena sangat njomplang)")
plt.yscale('log') # di setting log-scale soalnya fraudnya kedikit banget
plt.show()

# Kalau ini kita cek info harganya (Amount) dan waktunya
display(data_fraud[['Time', 'Amount', 'V1']].describe())

**Penjelasan Khusus Buat Credit Card Fraud:**  
Dari grafik *countplot* kita lihat sekeranjangnya transaksi perbankan (lebih dari 200 ribu lebih kelas 0), eh kasus fraud-nya cuma secuil (kelas 1). Inilah kenapa data ini disebut *imbalanced data banget*. Kondisi kayak gini nggak bisa dinilai bagus cuma dengan metrik *Accuracy*. Soalnya model bisa pura-pura nebak 'Nggak Ada Fraud' seharian, dan akurasinya bakal tetep di 99%. Jadi nanti untuk ngerjainnya kita harus perhatiin *Recall* sama nyisipin langkah buat nyeimbangin proporsinya kayak SMOTE.

---
## 2. Preprocessing & Bikin Data Training-Test

In [ ]:
# --------- Pisahin X dan y punya MADELON ---------
X_madelon = data_madelon.drop('kelas_target', axis=1)
y_madelon = data_madelon['kelas_target']

X_latih_mdln, X_uji_mdln, y_latih_mdln, y_uji_mdln = train_test_split(
    X_madelon, y_madelon, test_size=0.2, stratify=y_madelon, random_state=RANDOM_SEED
)

# --------- Pisahin X dan y punya Credit Card Fraud ---------
X_fraud = data_fraud.drop('Class', axis=1)
y_fraud = data_fraud['Class']

# PENTING: pake stratify biar pembagian persentase class fraud di train/test tetep terjaga keasliannya
X_latih_cc, X_uji_cc, y_latih_cc, y_uji_cc = train_test_split(
    X_fraud, y_fraud, test_size=0.2, stratify=y_fraud, random_state=RANDOM_SEED
)

print("Siap Latih! -> Jumlah Data MADELON (Latih):", X_latih_mdln.shape[0], ", (Uji):", X_uji_mdln.shape[0])
print("Siap Latih! -> Jumlah Data Fraud   (Latih):", X_latih_cc.shape[0], ", (Uji):", X_uji_cc.shape[0])

---
## 3 & 4. Setup Pipeline dan Parameter ML-nya

Karena ujian UAS minta perpaduan preprocessing:
1. **Data Madelon (Kolomnya Kebanyakan)**: Bakal di-treat pakai strategi buang-buangin fitur nggak penting pake Filter ANOVA (`SelectKBest`), terus dimensi datanya diteken / di-*squash* lagi pakai `PCA`.
2. **Credit Card Fraud (Terlalu Banyak Kelas Mayoritas)**: Bakal dipakein trik *SMOTE* buat memperbanyak tiruan data fraud secara artifisial tanpa membocorkan data.

In [ ]:
# Wadah nampung nilai rapor masing-masing model
rapor_model = []

# Fungsi ini dibuat sekalian buat nilai evaluasinya kelar di satu tempat
def cek_performa_model(model_dikemas, X_uji, y_aktual, nama_data, nama_model):
    y_tebakan = model_dikemas.predict(X_uji)
    
    # Ambil index-nya peluang positif (indeks kalau prediksinya angka 1)
    index_positif = np.where(model_dikemas.classes_ == 1)[0][0]
    uang_probabilitas = model_dikemas.predict_proba(X_uji)[:, index_positif]

    # Ini ngitung rapor sesuai permintaan Dosen
    akurasi = accuracy_score(y_aktual, y_tebakan)
    presisi = precision_score(y_aktual, y_tebakan, pos_label=1, zero_division=0)
    ingatan_recall = recall_score(y_aktual, y_tebakan, pos_label=1, zero_division=0)
    skor_f1 = f1_score(y_aktual, y_tebakan, pos_label=1, zero_division=0)
    nilai_auc = roc_auc_score(y_aktual, uang_probabilitas)
    
    rapor_model.append({
        'Dataset Target': nama_data,
        'Model Digunakan': nama_model,
        'Akurasi': f"{akurasi:.4f}",
        'Presisi': f"{presisi:.4f}",
        'Recall': f"{ingatan_recall:.4f}",
        'F1-Score': f"{skor_f1:.4f}",
        'AUC/ROC': f"{nilai_auc:.4f}"
    })
    
    print(f"\n> === Laporan Evaluasi Uji Buat [{nama_model}] ===")
    print(f"Accuracy  : {akurasi:.4f}")
    print(f"Precision : {presisi:.4f}")
    print(f"Recall    : {ingatan_recall:.4f}")
    print(f"F1 Measure: {skor_f1:.4f}")
    print(f"AUC       : {nilai_auc:.4f}")
    
    # Yuk sekalian munculin Matriks Tebakan
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    
    cm = confusion_matrix(y_aktual, y_tebakan)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0])
    axes[0].set_title(f'Tabel Kebingungan (CM) - {nama_model}')
    axes[0].set_ylabel('Fakta Asli')
    axes[0].set_xlabel('Tebakan Model')
    
    # Gambar kurva batas bawahnya 
    fpr, tpr, kemiringin = roc_curve(y_aktual, uang_probabilitas, pos_label=1)
    axes[1].plot(fpr, tpr, label=f'Rapor AUC = {nilai_auc:.2f}', color='red', lw=2)
    axes[1].plot([0,1],[0,1],'k--')
    axes[1].set_title(f'Kurva Area (ROC) - {nama_model}')
    axes[1].set_xlabel('Rate Salah Nebak Negatif')
    axes[1].set_ylabel('Rate Bener Nebak Positif')
    axes[1].legend(loc='lower right')
    
    plt.tight_layout()
    plt.show()

# Ini cross-validation settings nya
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
cv_cepet = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED) # di set jadi 3 aja buat CC biar komputernya gak capek

### Langsung Eksekusi Dataset Pertama: MADELON

In [ ]:
# -------- Skenario 1: Logistic Regression (+ Seleksi Fitur & PC) --------
print(">> Latih Logistic Regression buat data banyak dimensi...")

# Urutannya : standarin tipe nilai -> Saring 100 fitur terbaik -> Padatkan jadi 20 fitur -> Kasih ke LogReg
alur_logreg_mdln = ImbPipeline([
    ('normalisasi', StandardScaler()),
    ('filter_penting', SelectKBest(f_classif, k=100)), 
    ('padatin_dimensi', PCA(n_components=20)),         
    ('algo', LogisticRegression(random_state=RANDOM_SEED))
])

# Parameternya mau coba parameter penalti (C) ini
settingan_cari = {
    'algo__C': [0.1, 1, 10]
}

mesin_logreg_mdln = GridSearchCV(alur_logreg_mdln, settingan_cari, cv=cv, scoring='accuracy', n_jobs=-1)
mesin_logreg_mdln.fit(X_latih_mdln, y_latih_mdln)

cek_performa_model(mesin_logreg_mdln, X_uji_mdln, y_uji_mdln, "MADELON", "Logistic Regression (+KBest+PCA)")

In [ ]:
# -------- Skenario 2: Random Forest --------
print(">> Latih Ensembles Hutan (Random Forest) Khusus Dataset Lebar...")

alur_rf_mdln = ImbPipeline([
    ('normalisasi', StandardScaler()),
    ('filter_sekalian', SelectKBest(f_classif, k=50)), 
    ('algo', RandomForestClassifier(random_state=RANDOM_SEED))
])

settingan_rf = {
    'algo__n_estimators': [100, 200]
}

mesin_rf_mdln = GridSearchCV(alur_rf_mdln, settingan_rf, cv=cv, scoring='accuracy', n_jobs=-1)
mesin_rf_mdln.fit(X_latih_mdln, y_latih_mdln)

cek_performa_model(mesin_rf_mdln, X_uji_mdln, y_uji_mdln, "MADELON", "Random Forest (+KBest)")

### Lanjut Eksekusi Dataset Kedua: Credit Card Fraud (Imbalanced)

In [ ]:
# -------- Skenario 3: Model Pake SMOTE --------
print(">> Latih Data Miring: Logreg pakai senjata SMOTE...")

# Disini kunci penting pre-processing class imbalanced-nya berpadu
alur_logreg_cc = ImbPipeline([
    ('normalisasi', StandardScaler()),
    ('nambah_sintesis', SMOTE(random_state=RANDOM_SEED)),
    ('algo', LogisticRegression(random_state=RANDOM_SEED))
])

settingan_logreg_cc = {
    'algo__C': [0.1, 1]
}

# Perhatiin kita pakai scoring roc_auc, soalnya cari akurasi doang di imbalanced nipu gan
mesin_logreg_cc = GridSearchCV(alur_logreg_cc, settingan_logreg_cc, cv=cv_cepet, scoring='roc_auc', n_jobs=-1)
mesin_logreg_cc.fit(X_latih_cc, y_latih_cc)

cek_performa_model(mesin_logreg_cc, X_uji_cc, y_uji_cc, "Kartu Kredit Fraud", "LogReg (+SMOTE)")

In [ ]:
# -------- Skenario 4: Senjata Pamungkas Parameter Class-Weights --------
print(">> Latih Data Miring 2: Random Forest dengan auto class weight...")

# Kenapa paka class_weight='balanced'? Soalnya SMOTE kadang lamban banget kalau datanya >200ribu baris 
# Pilihan cerdas RF adalah bikin cost penalty (denda poin) berat kalo dia salah nebak Fraud.
alur_rf_cc = ImbPipeline([
    ('normalisasi', StandardScaler()),
    ('algo', RandomForestClassifier(class_weight='balanced', random_state=RANDOM_SEED))
])

settingan_rf_cc = {
    'algo__n_estimators': [50],
    'algo__max_depth': [5, 10]
}

mesin_rf_cc = GridSearchCV(alur_rf_cc, settingan_rf_cc, cv=cv_cepet, scoring='f1', n_jobs=-1)
mesin_rf_cc.fit(X_latih_cc, y_latih_cc)

cek_performa_model(mesin_rf_cc, X_uji_cc, y_uji_cc, "Kartu Kredit Fraud", "Random Forest (+Balanced Weights)")

---
## 5 & 6. Penjelasan Parameter & Pertunjukan Evaluasinya

Biar waktu presentasi UAS paham defisininya, ini pengingat cepatnya ya kawan :
- **Accuracy**: Rata-rata tebak bener dari si model (Dilarang dijadiin patokan utama buat data timpang/imbalanced).
- **Precision**: Kalau mesinnya bunyi sirine meramal *Ini Pelakunya (Fraud)*, berapa sih probabilitas di dunia nyata bahwa dia emang beneran pelakunya dan bukan orang lewat?
- **Recall**: Ini ibarat jaring ikan, makin tinggi jaring ini menangkap semua penipu *Fraud* yang harusnya ditangkap (Gak papa kadang nangkep salah, asal lolos nangkepnya minimal banget).
- **F1-Score**: Ya, metrik kompromi win-win solution antara Precision dan Recall. Ideal buat data imbalanced.
- **AUC**: Skor (makin nekat deket 1.0) yg nentuin *skill level* mesin buat memisahkan 'kriminal' sama 'anggota polos'.

In [ ]:
# Nah, ini tabel rangkuman biar gampang milih model terbaiknya kawan!
Tabel_Kesimpulan = pd.DataFrame(rapor_model)

print("======== REKAPITULASI SIAPA YANG TERBAIK ========")
display(Tabel_Kesimpulan.sort_values(by=['Dataset Target', 'AUC/ROC', 'F1-Score'], ascending=[True, False, False]))

---
## 7. Kesimpulan & Penutup Akademik untuk Presentasi UAS

Untuk menutup *Project Machine Learning Pipeline* kelompok kita, beginilah konklusinya:

**A. Eksplorasi Strategi di MADELON (Tipe High Dimensional)**
1. Kita tahu data ini *menyebalkan* gara-gara isinya 500 atribut dan sebagian besar adalah *noise* / variabel siluman pengalih (Probe). Kalau regresi logistic biasa nebak mentahan, pasti overfit & nyangkut. 
2. Solusinya? Kuncinya ada di Pre-Processing Filter & Reduksi. Teknik Filter `SelectKBest` buang parasitnya, & disusul kompresor dimensi andalan `PCA`. Setelah 500 ngerucut sisa 20 Komponen berharga, model sesimpel *Logistic Regression* aja performanya langsung meroket tajam di parameter yang seimbang.

**B. Eksplorasi Strategi Klasifikasi Kredit Imbalanced**
1. Secara teknis, jumlah fraud sangat jomplang (bisa dilihat di grafik EDA skala logaritma kita di atas).
2. Ada perpaduan proses untuk obatin ini. Penanganan andalan kita pakai **SMOTE** buat menyuntik bibit buatan secara algoritmis bagi Logistic Regresion.
3. Namun untuk si pohon sakti **Random Forest**, ketimbang di-smote yang boros CPU, kita pakai kunci parameter `class_weight='balanced'`. Algoritma RF bakal dihukum penalti/rugi gede kalau tebakannya membiarkan *Fraud* lewat. Akhirnya Recall bisa naik tajam menyelamatkan harta institusi tanpa terlalu ngerusak precision!

**C. Siapakah Sang Juara Bertahan?**
- 🏆 Buat kawan **MADELON**: **Logistic Regression (PCA + SelectKBest)** atau **Random Forest** punya nilai yang beda tipis namun sangat konsisten setelah dibersihkan noisenya. 
- 🏆 Buat pertarungan **CC Fraud**: **Random Forest with Balanced Penalty Weights** jadi jagoannya. Karena si RF bisa memblokir fitur noise *non-linear* sambil mengutamakan *Recall* yg memompa F1-score/AUC. 

*Sekian kawan, program berjalan lancar tanpa kendala. Boleh tepuk tangan untuk eksekusinya!*